## Fix Author Profile full_name

Many author profiles have a corrupted `full_name` — set from a single outlier work's `raw_author_name` rather than the dominant form across the profile's works. Phase 2 of `split-authors-name-parser` would wrongly split hundreds of thousands of correctly-assigned works from these profiles.

Examples:
- A5110822529: `full_name`="MARY E. NOLAN" (1 work), but 301/302 works are "Mike Nolan"
- A5020500648: `full_name`="Heath M. Burton" (4 works), but 987/1001 works are "Howard Burton"
- A5109158685: `full_name`="Gabriel Couto Thurler Klein" (1 work), but 565/566 works are "George Klein"

This notebook replaces `full_name` with the most representative raw_author_name (most frequent full form with matching dominant parsed_first) whenever the current `full_name` is an outlier by >=80% dominance and the author has >=5 works.

Prerequisite for `SplitOvermergedAuthorsPhase2.ipynb`. See `oxjobs/working/fix-author-full-names/` for job context.

### Cell 1: Build candidates — authors with corrupted full_name

For every author: aggregate parsed_first frequencies across all works. If the dominant parsed_first (>=80% of works, >=5 total works) differs from `parsed(full_name).first`, this profile is a candidate. Also compute the best new full_name — the most frequent raw_author_name whose parsed_first matches the dominant first, preferring longest non-initial forms.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_fullname_fix_candidates AS
WITH profile_parsed AS (
  SELECT oa.id AS author_id, oa.full_name AS current_full_name,
    oa.display_name, oa.works_count,
    an.parsed_name.first AS current_first
  FROM openalex.authors.openalex_authors oa
  JOIN openalex.authors.author_names an ON oa.full_name = an.raw_author_name
  WHERE oa.works_count >= 5
),
work_raws AS (
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n
  FROM openalex.works.work_authors wa
  WHERE wa.author_id IS NOT NULL
  GROUP BY wa.author_id, wa.raw_author_name
),
work_raws_parsed AS (
  SELECT wr.author_id, wr.raw_author_name, wr.n,
    an.parsed_name.first AS raw_first
  FROM work_raws wr
  JOIN openalex.authors.author_names an ON wr.raw_author_name = an.raw_author_name
  WHERE an.parsed_name.first IS NOT NULL
    AND LENGTH(an.parsed_name.first) > 2
    AND an.parsed_name.first NOT LIKE '%.%'
),
-- Sum by (author_id, parsed_first)
by_first AS (
  SELECT author_id, raw_first, SUM(n) AS n_works_for_first
  FROM work_raws_parsed
  GROUP BY author_id, raw_first
),
-- Dominant first per author
dominant AS (
  SELECT author_id, raw_first AS dominant_first, n_works_for_first AS dominant_works,
    SUM(n_works_for_first) OVER (PARTITION BY author_id) AS total_parseable_works
  FROM by_first
  QUALIFY ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY n_works_for_first DESC) = 1
),
-- Among raw_author_names matching the dominant first, pick the best full form:
-- prefer non-initial forms, then most frequent, then longest
best_raw AS (
  SELECT d.author_id, wrp.raw_author_name AS new_full_name, wrp.n AS new_full_name_works
  FROM dominant d
  JOIN work_raws_parsed wrp ON wrp.author_id = d.author_id AND wrp.raw_first = d.dominant_first
  WHERE LENGTH(wrp.raw_author_name) > 3
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY d.author_id
    ORDER BY wrp.n DESC, LENGTH(wrp.raw_author_name) DESC, wrp.raw_author_name
  ) = 1
)
SELECT pp.author_id, pp.current_full_name, pp.display_name, pp.works_count,
  pp.current_first, d.dominant_first, d.dominant_works, d.total_parseable_works,
  br.new_full_name, br.new_full_name_works,
  (d.dominant_works * 1.0 / d.total_parseable_works) AS dominance_ratio
FROM profile_parsed pp
JOIN dominant d ON d.author_id = pp.author_id
JOIN best_raw br ON br.author_id = pp.author_id
WHERE d.dominant_first != pp.current_first
  AND d.dominant_works * 1.0 / d.total_parseable_works >= 0.80
  AND d.total_parseable_works >= 5
  AND br.new_full_name != pp.current_full_name

### Cell 2: Validation statistics

In [ ]:
SELECT
  COUNT(*) AS total_fix_candidates,
  PERCENTILE_APPROX(works_count, ARRAY(0.25, 0.5, 0.75, 0.95)) AS works_count_pctiles,
  PERCENTILE_APPROX(dominance_ratio, ARRAY(0.5, 0.9, 0.99)) AS dominance_pctiles,
  SUM(CASE WHEN dominance_ratio >= 0.95 THEN 1 ELSE 0 END) AS at_least_95pct,
  SUM(CASE WHEN dominance_ratio >= 0.90 THEN 1 ELSE 0 END) AS at_least_90pct,
  SUM(CASE WHEN dominance_ratio >= 0.80 THEN 1 ELSE 0 END) AS at_least_80pct
FROM openalex.authors.author_fullname_fix_candidates

### Cell 3: Spot-check sample (manual review)

In [ ]:
SELECT author_id, current_full_name, new_full_name, display_name,
  current_first, dominant_first, dominant_works, total_parseable_works,
  ROUND(dominance_ratio, 2) AS dominance, works_count
FROM openalex.authors.author_fullname_fix_candidates
ORDER BY RAND()
LIMIT 50

### Cell 4: Create audit log (for rollback)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_fullname_fix_log AS
SELECT author_id, current_full_name AS previous_full_name, new_full_name,
  current_first AS previous_first, dominant_first,
  dominant_works, total_parseable_works, dominance_ratio,
  current_timestamp() AS fix_timestamp
FROM openalex.authors.author_fullname_fix_candidates

### Cell 5: Execute the fix

**WARNING**: This overwrites `full_name` on `openalex_authors`. Verify cells 2–3 before running.

In [ ]:
MERGE INTO openalex.authors.openalex_authors AS target
USING openalex.authors.author_fullname_fix_candidates AS source
ON target.id = source.author_id
WHEN MATCHED THEN
  UPDATE SET
    target.full_name = source.new_full_name,
    target.updated_date = current_timestamp()

### Cell 6: Post-fix verification

In [ ]:
SELECT
  COUNT(*) AS updated_profiles,
  SUM(CASE WHEN oa.full_name = src.new_full_name THEN 1 ELSE 0 END) AS full_name_matches_new,
  SUM(CASE WHEN oa.full_name = src.previous_full_name THEN 1 ELSE 0 END) AS full_name_still_old
FROM openalex.authors.openalex_authors oa
JOIN openalex.authors.author_fullname_fix_log src ON src.author_id = oa.id

### Cell 7: Rollback template (use only for systematic FP discovered)

```sql
MERGE INTO openalex.authors.openalex_authors AS target
USING openalex.authors.author_fullname_fix_log AS source
ON target.id = source.author_id
WHEN MATCHED THEN
  UPDATE SET target.full_name = source.previous_full_name
```

Add a `WHERE` clause inside the USING subquery to restrict rollback to a specific class. Don't run as-is — it reverses all fixes.